# pip install sqlite3

In [1]:
import sqlite3  # Lets python connect to and query SQLite database files.
import pandas as pd # Used for working with data in table form

In [2]:
# Define the path to the SQLite database file.
db_path = "../data/raw/MIG_Cement_Records.db"

# open a connection to the SQLite database
conn = sqlite3.connect(db_path)

# Create a cursur object, which is used to execute SQL commands
cursor = conn.cursor()

In [3]:
# Query sqlite_master to list all tables in the database
tables = cursor.execute(
  "SELECT name FROM sqlite_master WHERE type='table';"
).fetchall()

# Print the list of tables found in the database.
print("Tables:", tables)

Tables: [('Sites',), ('CementTypes',), ('Operations',)]


In [4]:
# Look through each table name
for table_name, in tables:
  print(f"===={table_name}====")

  # PRAGMA returns the structure
  schema = cursor.execute(
    f"PRAGMA table_info({table_name});"
  ).fetchall()
  
  display(schema) # in jupyter display shows the output more cleanly than print()

====Sites====


[(0, 'site_id', 'TEXT', 0, None, 1),
 (1, 'region', 'TEXT', 0, None, 0),
 (2, 'silo_capacity', 'INTEGER', 0, None, 0),
 (3, 'behavior', 'TEXT', 0, None, 0)]

====CementTypes====


[(0, 'cement_type', 'TEXT', 0, None, 1)]

====Operations====


[(0, 'date', 'TEXT', 0, None, 0),
 (1, 'site_id', 'TEXT', 0, None, 0),
 (2, 'cement_type', 'TEXT', 0, None, 0),
 (3, 'planned_pour_tonnes', 'REAL', 0, None, 0),
 (4, 'consumed_tonnes', 'REAL', 0, None, 0),
 (5, 'opening_inventory_tonnes', 'REAL', 0, None, 0),
 (6, 'deliveries_tonnes', 'REAL', 0, None, 0),
 (7, 'closing_inventory_tonnes', 'REAL', 0, None, 0),
 (8, 'rain_mm', 'REAL', 0, None, 0),
 (9, 'avg_temp_c', 'REAL', 0, None, 0),
 (10, 'silo_capacity', 'INTEGER', 0, None, 0)]

In [5]:
# Loop through each table
for table_name, in tables:
  print(f"===={table_name}====")

  # display the sample rows to understand the kind of data each table contains
  rows = cursor.execute(
    f"SELECT * FROM {table_name} LIMIT 5;"
  ).fetchall()
  display('Sample rows:', rows)

====Sites====


'Sample rows:'

[('SITE_001', 'North', 448, 'aggressive'),
 ('SITE_002', 'South', 288, 'conservative'),
 ('SITE_003', 'East', 314, 'aggressive'),
 ('SITE_004', 'South', 472, 'conservative'),
 ('SITE_005', 'South', 230, 'aggressive')]

====CementTypes====


'Sample rows:'

[('CEM_I',), ('CEM_II',), ('CEM_III',)]

====Operations====


'Sample rows:'

[('2022-01-01',
  'SITE_001',
  'CEM_II',
  43.18,
  34.54,
  52.56,
  45.83,
  63.85,
  3.4,
  -3.1,
  448),
 ('2022-01-02',
  'SITE_001',
  'CEM_I',
  45.26,
  45.26,
  63.85,
  19.97,
  38.56,
  3.23,
  14.28,
  448),
 ('2022-01-03',
  'SITE_001',
  'CEM_III',
  38.69,
  38.69,
  38.56,
  47.19,
  47.06,
  2.64,
  6.4,
  448),
 ('2022-01-04',
  'SITE_001',
  'CEM_I',
  33.16,
  33.16,
  47.06,
  18.74,
  32.64,
  8.25,
  14.23,
  448),
 ('2022-01-05',
  'SITE_001',
  'CEM_III',
  56.88,
  47.04,
  32.64,
  14.4,
  0.0,
  2.69,
  8.97,
  448)]

In [6]:
# Combine data from Operations with Sites
query = """
SELECT
  o.date,
  o.site_id,
  s.region,
  s.behavior,
  o.cement_type,
  o.planned_pour_tonnes,
  o.consumed_tonnes,
  o.opening_inventory_tonnes,
  o.deliveries_tonnes,
  o.closing_inventory_tonnes,
  o.rain_mm,
  o.avg_temp_c,
  o.silo_capacity
FROM Operations o
JOIN Sites s ON o.site_id = s.site_id
"""

In [7]:
# Run the query through the connection and load it into pandas DataFrame df
df = pd.read_sql_query(
  query, 
  conn
)

In [8]:
# Display the first 5 rows of the DataFrame
df.head()

,date,site_id,region,behavior,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity
0,2022-01-01,SITE_001,North,aggressive,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448
1,2022-01-02,SITE_001,North,aggressive,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448
2,2022-01-03,SITE_001,North,aggressive,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448
3,2022-01-04,SITE_001,North,aggressive,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448
4,2022-01-05,SITE_001,North,aggressive,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448


In [9]:
# Close connection to the SQLite database after completing queries
conn.close()

In [10]:
# Show the technical summary of the DataFrame
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32880 entries, 0 to 32879
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      32880 non-null  object 
 1   site_id                   32880 non-null  object 
 2   region                    32880 non-null  object 
 3   behavior                  32880 non-null  object 
 4   cement_type               32880 non-null  object 
 5   planned_pour_tonnes       32880 non-null  float64
 6   consumed_tonnes           32880 non-null  float64
 7   opening_inventory_tonnes  32880 non-null  float64
 8   deliveries_tonnes         32880 non-null  float64
 9   closing_inventory_tonnes  32880 non-null  float64
 10  rain_mm                   32880 non-null  float64
 11  avg_temp_c                32880 non-null  float64
 12  silo_capacity             32880 non-null  int64  
dtypes: float64(7), int64(1), object(5)
memory usage: 3.3+ MB


In [11]:
# Convert date column from object to proper datetime format
df["date"] =pd.to_datetime(df["date"])

In [12]:
# Check for any missing values
df.isnull().sum()

date                        0
site_id                     0
region                      0
behavior                    0
cement_type                 0
planned_pour_tonnes         0
consumed_tonnes             0
opening_inventory_tonnes    0
deliveries_tonnes           0
closing_inventory_tonnes    0
rain_mm                     0
avg_temp_c                  0
silo_capacity               0
dtype: int64

In [16]:
# convert DataFrame column name to regular python list for clearer view 
df.columns.tolist()

['date',
 'site_id',
 'region',
 'behavior',
 'cement_type',
 'planned_pour_tonnes',
 'consumed_tonnes',
 'opening_inventory_tonnes',
 'deliveries_tonnes',
 'closing_inventory_tonnes',
 'rain_mm',
 'avg_temp_c',
 'silo_capacity']

In [18]:
# create a list of columns that contains numeric values
numeric_cols = [
  'planned_pour_tonnes',
  'consumed_tonnes', 
  'opening_inventory_tonnes',    
  'deliveries_tonnes',
  'closing_inventory_tonnes', 
  'rain_mm', 
  'avg_temp_c',                  
  'silo_capacity'
]

# cheeck each numeric column for values less than 0
(df[numeric_cols] < 0).sum()

planned_pour_tonnes            0
consumed_tonnes                0
opening_inventory_tonnes       0
deliveries_tonnes              0
closing_inventory_tonnes       0
rain_mm                        0
avg_temp_c                  4551
silo_capacity                  0
dtype: int64

In [19]:
# automated code to select numeric values
numeric_colss = df.select_dtypes(include='number').columns
(df[numeric_colss] < 0).sum()

planned_pour_tonnes            0
consumed_tonnes                0
opening_inventory_tonnes       0
deliveries_tonnes              0
closing_inventory_tonnes       0
rain_mm                        0
avg_temp_c                  4551
silo_capacity                  0
dtype: int64